## 26. Validation Suite — Discrimination & Calibration

In [ ]:
# ============================================================
# CELL 1 — ROC-AUC, Gini, KS for both models, plotted together
# ============================================================
from sklearn.metrics import roc_auc_score, roc_curve

# LightGBM predictions (already computed during training, reuse if available,
# otherwise recompute here)
lgb_probs = calibrated_lgb.predict_proba(X_test_lgb)[:, 1]
auc_lgb = roc_auc_score(y_test_lgb, lgb_probs)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test_lgb, lgb_probs)
ks_lgb = max(tpr_lgb - fpr_lgb)
gini_lgb = 2 * auc_lgb - 1

# Logistic scorecard predictions
lr_probs = calibrated_model.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, lr_probs)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
ks_lr = max(tpr_lr - fpr_lr)
gini_lr = 2 * auc_lr - 1

print("--- Model Performance Summary ---")
print(f"{'Metric':<10} {'LightGBM':>12} {'Logistic':>12}")
print(f"{'AUC':<10} {auc_lgb:>12.4f} {auc_lr:>12.4f}")
print(f"{'Gini':<10} {gini_lgb:>12.4f} {gini_lr:>12.4f}")
print(f"{'KS':<10} {ks_lgb:>12.4f} {ks_lr:>12.4f}")

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_lgb, tpr_lgb, label=f'LightGBM (AUC={auc_lgb:.3f})', color='#2ecc71', linewidth=2)
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Scorecard (AUC={auc_lr:.3f})', color='#3498db', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 2 — Calibration plot: predicted PD vs actual default rate by decile
# ============================================================
def calibration_by_decile(y_true, y_prob, n_bins=10):
    df_cal = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    df_cal['decile'] = pd.qcut(df_cal['y_prob'], q=n_bins, duplicates='drop')
    summary = df_cal.groupby('decile').agg(
        predicted_pd=('y_prob', 'mean'),
        actual_default_rate=('y_true', 'mean'),
        count=('y_true', 'count')
    ).reset_index(drop=True)
    return summary

cal_lgb = calibration_by_decile(y_test_lgb, lgb_probs)
cal_lr = calibration_by_decile(y_test, lr_probs)

print("--- LightGBM Calibration by Decile ---")
print(cal_lgb.round(4))
print("\n--- Logistic Scorecard Calibration by Decile ---")
print(cal_lr.round(4))

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(cal_lgb['predicted_pd'], cal_lgb['actual_default_rate'], 'o-',
        color='#2ecc71', label='LightGBM', markersize=8)
ax.plot(cal_lr['predicted_pd'], cal_lr['actual_default_rate'], 'o-',
        color='#3498db', label='Logistic Scorecard', markersize=8)
ax.set_xlabel('Predicted PD (mean per decile)')
ax.set_ylabel('Actual Default Rate (per decile)')
ax.set_title('Calibration Plot: Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 3 — Hosmer-Lemeshow test (formal calibration test)
# ============================================================
from scipy.stats import chi2

def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    df_hl = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    df_hl['decile'] = pd.qcut(df_hl['y_prob'], q=n_bins, duplicates='drop')

    grouped = df_hl.groupby('decile').agg(
        observed=('y_true', 'sum'),
        expected=('y_prob', 'sum'),
        n=('y_true', 'count')
    )
    grouped['expected_nonevent'] = grouped['n'] - grouped['expected']
    grouped['observed_nonevent'] = grouped['n'] - grouped['observed']

    hl_stat = (
        ((grouped['observed'] - grouped['expected']) ** 2 / grouped['expected']) +
        ((grouped['observed_nonevent'] - grouped['expected_nonevent']) ** 2 / grouped['expected_nonevent'])
    ).sum()

    dof = len(grouped) - 2
    p_value = 1 - chi2.cdf(hl_stat, dof)
    return hl_stat, p_value, dof

hl_stat_lgb, p_lgb, dof_lgb = hosmer_lemeshow_test(y_test_lgb, lgb_probs)
hl_stat_lr, p_lr, dof_lr = hosmer_lemeshow_test(y_test, lr_probs)

print("--- Hosmer-Lemeshow Test ---")
print(f"LightGBM:  chi2={hl_stat_lgb:.2f}, df={dof_lgb}, p-value={p_lgb:.4f}")
print(f"Logistic:  chi2={hl_stat_lr:.2f}, df={dof_lr}, p-value={p_lr:.4f}")
print(f"\nInterpretation: p > 0.05 suggests the model is well-calibrated (fail to")
print(f"reject the null of good fit). p < 0.05 suggests significant miscalibration.")

In [ ]:
# ============================================================
# CELL 4 — Brier score
# ============================================================
from sklearn.metrics import brier_score_loss

brier_lgb = brier_score_loss(y_test_lgb, lgb_probs)
brier_lr = brier_score_loss(y_test, lr_probs)

print("--- Brier Score (lower is better) ---")
print(f"LightGBM:  {brier_lgb:.4f}")
print(f"Logistic:  {brier_lr:.4f}")

In [ ]:
# ============================================================
# CELL 5 — Population Stability Index (PSI)
# ============================================================
# PSI compares the predicted-score distribution between two populations --
# typically "training/development" vs "current/test" -- to check if the
# population being scored has drifted from what the model was built on.

def calculate_psi(expected, actual, n_bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    expected_pct = np.clip(expected_pct, 1e-6, None)
    actual_pct = np.clip(actual_pct, 1e-6, None)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

# Compare training-set scores vs test-set scores as a proxy for "expected vs actual"
train_probs_lgb = calibrated_lgb.predict_proba(X_train_lgb)[:, 1]
psi_lgb = calculate_psi(train_probs_lgb, lgb_probs)

train_probs_lr = calibrated_model.predict_proba(scaler.transform(X_train))[:, 1]
psi_lr = calculate_psi(train_probs_lr, lr_probs)

print("--- Population Stability Index (train scores vs test scores) ---")
print(f"LightGBM PSI:  {psi_lgb:.4f}")
print(f"Logistic PSI:  {psi_lr:.4f}")
print(f"\nInterpretation: PSI < 0.1 = stable, 0.1-0.2 = moderate shift, > 0.2 = unstable.")